# 02 · Hình R0 từ map tác giả (đợt 1) — 0 đồng GPU

Chỉ đọc `dire/` + `images/`. **Không** cần ADM, không cần checkpoint của mình,
không cần GPU. Nếu chỉ làm được một đợt thì làm đợt này.

Nhãn: map là **của tác giả** ⇒ `R0`. Nếu bạn tự tính map trên ảnh của tác giả
thì nhãn là `R1+R2` — đổi `--label`, đừng để mặc định.

In [ ]:
# Chạy từ thư mục gốc repo. Notebook CHỈ gọi tools/figures.py —
# không tự viết lại logic vẽ, để hình trong notebook và hình trong báo cáo
# luôn là cùng một hình.
import json, os, subprocess, sys
from pathlib import Path

ROOT = Path.cwd()
assert (ROOT / 'tools' / 'figures.py').exists(), 'hãy chạy notebook từ thư mục gốc repo'
FIG = Path(os.environ.get('DIRE_FIG_OUT', 'figs'))

def fig_cmd(*args):
    """Chạy tools/figures.py, in lệnh rồi in đầu ra — lệnh in ra là lệnh tái lập."""
    cmd = [sys.executable, str(ROOT / 'tools' / 'figures.py'), *map(str, args)]
    print('$', ' '.join(cmd[1:]))
    r = subprocess.run(cmd, cwd=ROOT, capture_output=True, text=True)
    print(r.stdout.strip() or r.stderr.strip())
    r.check_returncode()
    return r

def sidecar(name):
    """Đọc sidecar: nguồn, n, nhãn. Không có sidecar = hình không kiểm được."""
    return json.loads((FIG / 'data' / f'{name}.json').read_text(encoding='utf-8'))

In [ ]:
MAPS = Path(os.environ.get('DIRE_FIG_MAPS', 'data/maps-r0'))
IMAGES = Path(os.environ.get('DIRE_FIG_IMAGES', 'data/dire-zero-norm'))
SPLIT = os.environ.get('DIRE_FIG_SPLIT', 'test')
GENERATOR = os.environ.get('DIRE_FIG_GENERATOR', 'adm')
print('maps:', MAPS, '| images:', IMAGES, '| split:', SPLIT, '| gen:', GENERATOR)

## Bước 0 — kiểm ghép cặp TRƯỚC khi vẽ

Giả định ngầm của hình teaser là member thứ *i* của `images/<gen>.tar.gz` ứng
với member thứ *i* của `dire/<gen>.tar.gz`. **Chưa ai chứng minh.** Nếu sai,
hình sẽ ghép map của ảnh A lên ảnh B — vẫn ra hình, vẫn thuyết phục, và sai.

Đổi hai đường dẫn dưới đây thành tar thật của bạn rồi chạy ô kiểm.

In [ ]:
# Đặt hai biến môi trường này (đường dẫn trên máy bạn), rồi chạy ô kiểm.
# IMG_TAR = .../DiffusionForensics/images/test/lsun_bedroom/adm.tar.gz
# MAP_TAR = .../DiffusionForensics/dire/test/lsun_bedroom/lsun_bedroom/adm.tar.gz
IMG_TAR = os.environ.get('IMG_TAR')
MAP_TAR = os.environ.get('MAP_TAR')
IMG_TAR = Path(IMG_TAR) if IMG_TAR else None
MAP_TAR = Path(MAP_TAR) if MAP_TAR else None

if IMG_TAR and MAP_TAR:
    import importlib.util
    spec = importlib.util.spec_from_file_location('figs_tool', ROOT / 'tools' / 'figures.py')
    tool = importlib.util.module_from_spec(spec); spec.loader.exec_module(tool)
    res = tool.pair_tars(IMG_TAR, MAP_TAR)
    print(json.dumps(res, ensure_ascii=False, indent=2))
    assert res['ok'], 'KHÔNG ghép theo thứ tự được — dùng đường recons/ (xem figures-plan §2.1)'
else:
    print('chưa đặt IMG_TAR/MAP_TAR — bỏ qua bước kiểm (F1 ghép theo tên stem, không theo thứ tự)')

## F5 — noise pattern + phổ tần số + phổ radial

In [ ]:
fig_cmd('f5-noise', '--out', FIG, '--maps', MAPS, '--split', SPLIT,
        '--generator', GENERATOR, '--n', 3, '--label', 'R0')

In [ ]:
sc = sidecar('f5-noise')
print('nhãn:', sc['claim_label'], '| n:', sc['n'], '| nguồn:', sc['sources']['maps'])
assert sc['claim_label'] == 'R0'

## F1 — teaser: ảnh gốc | DIRE

Ô này **cần cả ảnh và map**, cùng tên stem. Không có cặp khớp thì dừng —
đó là hàng rào, không phải lỗi cấu hình.

In [ ]:
if (IMAGES / SPLIT / GENERATOR).is_dir():
    fig_cmd('f1-teaser', '--out', FIG, '--images', IMAGES, '--maps', MAPS,
            '--split', SPLIT, '--generator', GENERATOR, '--n', 3, '--label', 'R0')
else:
    print('chưa có ảnh đã ingest ở', IMAGES / SPLIT / GENERATOR, '— bỏ qua F1')